# Walkie Data 实验 notebook

目标：通过 `data/download.py` 下载多个 Hugging Face 数据集，抽取子集训练 `byte_bpe` tokenizer，统计 token 总量和各数据集占比，并把数据切成 Walkie 两阶段退火预训练可直接读取的 `main.bin` / `anneal.bin`。

请先在配置单元填写 `DATASETS`。高质量退火数据集把 `stage` 设为 `anneal`，大规模普通代码语料设为 `main`，不确定的设为 `auto`。

## 1. 环境导入

定位仓库根目录，导入现有下载、文本迭代和 tokenizer 工具。当前 `data/download.py` 文档写的是 `download`，如果实际函数名是临时的 `c`，这里会自动兼容。

In [ ]:
from __future__ import annotations

import hashlib
import importlib
import json
import random
import re
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import numpy as np
from tqdm.auto import tqdm

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.tokenizer import build_tokenizer, load_tokenizer
from data.encode import iter_texts

download_mod = importlib.import_module("data.download")
DOWNLOAD_FN = getattr(download_mod, "download", None) or getattr(download_mod, "c", None)
if DOWNLOAD_FN is None:
    raise RuntimeError("data/download.py 中没有可调用的 download(...) 或兼容函数 c(...)")

print(f"ROOT = {ROOT}")
print(f"Using downloader: data.download.{DOWNLOAD_FN.__name__}")

## 2. 实验配置

`DATASETS` 默认留空，避免误下载。每个数据集支持：`repo_id`、`subset_name`、`split`、`text_field`、`num_shards`、`allow_patterns`、`ignore_patterns`、`max_samples`、`max_chars`、`stage`、`quality_hint`。

默认导出 token 比例采用代码预训练常见做法：`main≈85%`、`anneal≈15%`。训练脚本仍按最后 20% step 切换到退火数据，因此 anneal 高质量数据会被更频繁复用。

In [ ]:
OUTPUT_DIR = ROOT / "data/cache/walkie_code"
SOURCES_DIR = OUTPUT_DIR / "sources"
INTERMEDIATE_DIR = OUTPUT_DIR / "intermediate"
TOKENIZER_PATH = OUTPUT_DIR / "tokenizer.json"
SAMPLES_JSONL = INTERMEDIATE_DIR / "filtered_samples.jsonl"
SPLIT_MANIFEST_JSONL = OUTPUT_DIR / "split_manifest.jsonl"
MAIN_BIN = OUTPUT_DIR / "main.bin"
ANNEAL_BIN = OUTPUT_DIR / "anneal.bin"
META_PATH = OUTPUT_DIR / "data_meta.json"

HF_ENDPOINT = "https://hf-mirror.com"  # 不需要镜像时改成 None
HF_TOKEN = None
MAX_WORKERS = 8
RANDOM_SEED = 42

TOKENIZER_KIND = "byte_bpe"
VOCAB_SIZE = 65_536
TOKENIZER_TRAIN_MAX_CHARS = 50_000_000
TOKENIZER_TRAIN_MAX_CHARS_PER_DATASET = 15_000_000
FORCE_RETRAIN_TOKENIZER = False
ADD_EOS = True

MIN_TEXT_CHARS = 80
MAX_TEXT_CHARS = 250_000
MIN_STORE_QUALITY = 0.05
FORCE_RESCAN = False

MAIN_TOKEN_RATIO = 0.85
ANNEAL_TOKEN_RATIO = 0.15
QUALITY_SCORE_FOR_ANNEAL = 0.62
AUTO_ANNEAL_TOP_FRACTION = 0.20
FORCE_EXPORT_BINS = True

DATASETS: list[dict[str, Any]] = [
    # 示例：复制后替换为真实 HF dataset。
    # {
    #     "name": "large_python_main",
    #     "repo_id": "owner/python-code-dataset",
    #     "subset_name": None,
    #     "split": "train",
    #     "text_field": "content",
    #     "num_shards": 2,
    #     "allow_patterns": None,
    #     "ignore_patterns": None,
    #     "max_samples": None,
    #     "max_chars": None,
    #     "stage": "main",
    #     "quality_hint": 0.55,
    # },
]

for path in (OUTPUT_DIR, SOURCES_DIR, INTERMEDIATE_DIR):
    path.mkdir(parents=True, exist_ok=True)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"OUTPUT_DIR = {OUTPUT_DIR}")
if not DATASETS:
    print("DATASETS 仍为空：请先填写数据集清单，然后继续执行。")

## 3. 下载多个 HF 数据集

In [ ]:
def safe_name(value: str) -> str:
    value = value.replace("/", "__")
    value = re.sub(r"[^0-9A-Za-z_.-]+", "_", value)
    return value.strip("._-") or "dataset"


def normalize_dataset_cfg(ds: dict[str, Any]) -> dict[str, Any]:
    if not ds.get("repo_id"):
        raise ValueError(f"数据集缺少 repo_id: {ds}")
    out = dict(ds)
    out.setdefault("name", safe_name(str(out["repo_id"])))
    out.setdefault("subset_name", None)
    out.setdefault("split", "train")
    out.setdefault("text_field", None)
    out.setdefault("num_shards", None)
    out.setdefault("allow_patterns", None)
    out.setdefault("ignore_patterns", None)
    out.setdefault("max_samples", None)
    out.setdefault("max_chars", None)
    out.setdefault("stage", "auto")
    out.setdefault("quality_hint", 0.5)
    if out["stage"] not in {"main", "anneal", "auto"}:
        raise ValueError(f"stage 必须是 main/anneal/auto: {out['stage']}")
    out["cache_dir"] = SOURCES_DIR / safe_name(out["name"])
    return out


def write_dataset_meta(ds: dict[str, Any]) -> None:
    meta = {
        "repo_id": ds["repo_id"],
        "subset_name": ds.get("subset_name"),
        "split": ds.get("split", "train"),
        "text_field": ds.get("text_field"),
        "max_samples": ds.get("max_samples"),
        "max_chars": ds.get("max_chars"),
        "stage": ds.get("stage", "auto"),
        "quality_hint": ds.get("quality_hint", 0.5),
    }
    path = Path(ds["cache_dir"]) / "data_meta.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")


def download_dataset(ds: dict[str, Any]) -> Path:
    kwargs = dict(
        repo_id=ds["repo_id"],
        local_dir=ds["cache_dir"],
        repo_type="dataset",
        subset_name=ds.get("subset_name"),
        num_shards=ds.get("num_shards"),
        hf_endpoint=HF_ENDPOINT,
        token=HF_TOKEN,
        max_workers=MAX_WORKERS,
    )
    if ds.get("allow_patterns") is not None:
        kwargs["allow_patterns"] = ds["allow_patterns"]
    if ds.get("ignore_patterns") is not None:
        kwargs["ignore_patterns"] = ds["ignore_patterns"]
    snapshot_dir = DOWNLOAD_FN(**kwargs)
    write_dataset_meta(ds)
    return Path(snapshot_dir)


DATASETS = [normalize_dataset_cfg(ds) for ds in DATASETS]
if not DATASETS:
    raise ValueError("DATASETS 为空：请先填写数据集清单。")

for ds in DATASETS:
    print(f"\n=== {ds['name']} ({ds['repo_id']}) stage={ds['stage']} ===")
    snapshot = download_dataset(ds)
    print(f"local snapshot: {snapshot}")

## 4. 抽取子集训练 tokenizer

In [ ]:
def iter_dataset_texts(ds: dict[str, Any], *, max_samples: int | None = None, max_chars: int | None = None) -> Iterable[str]:
    yield from iter_texts(
        ds["cache_dir"],
        split=ds.get("split", "train"),
        text_field=ds.get("text_field"),
        max_samples=max_samples if max_samples is not None else ds.get("max_samples"),
        max_chars=max_chars if max_chars is not None else ds.get("max_chars"),
    )


def collect_tokenizer_corpus() -> list[str]:
    corpus: list[str] = []
    total_chars = 0
    order = list(DATASETS)
    random.Random(RANDOM_SEED).shuffle(order)
    for ds in order:
        ds_chars = 0
        pbar = tqdm(desc=f"tokenizer sample: {ds['name']}", unit="chars")
        for text in iter_dataset_texts(ds, max_chars=TOKENIZER_TRAIN_MAX_CHARS_PER_DATASET):
            text = text.strip()
            if not text:
                continue
            remaining_total = TOKENIZER_TRAIN_MAX_CHARS - total_chars
            remaining_ds = TOKENIZER_TRAIN_MAX_CHARS_PER_DATASET - ds_chars
            take = min(len(text), remaining_total, remaining_ds)
            if take <= 0:
                break
            corpus.append(text[:take])
            total_chars += take
            ds_chars += take
            pbar.update(take)
            if total_chars >= TOKENIZER_TRAIN_MAX_CHARS or ds_chars >= TOKENIZER_TRAIN_MAX_CHARS_PER_DATASET:
                break
        pbar.close()
        print(f"{ds['name']}: tokenizer chars={ds_chars:,}")
        if total_chars >= TOKENIZER_TRAIN_MAX_CHARS:
            break
    print(f"total tokenizer train chars={total_chars:,}")
    return corpus


if TOKENIZER_PATH.exists() and not FORCE_RETRAIN_TOKENIZER:
    tokenizer = load_tokenizer(TOKENIZER_PATH)
    print(f"[tokenizer] loaded: {TOKENIZER_PATH} vocab={tokenizer.vocab_size}")
else:
    corpus = collect_tokenizer_corpus()
    if not corpus:
        raise RuntimeError("没有收集到 tokenizer 训练文本，请检查 DATASETS/text_field/split。")
    tok_cls = type(build_tokenizer(TOKENIZER_KIND))
    tokenizer = tok_cls.train(corpus, vocab_size=VOCAB_SIZE, verbose=True)
    tokenizer.save(TOKENIZER_PATH)
    print(f"[tokenizer] saved: {TOKENIZER_PATH} vocab={tokenizer.vocab_size}")

DTYPE = np.dtype(np.uint16 if tokenizer.vocab_size < 65_536 else np.uint32)
print(f"bin dtype = {DTYPE}")

## 5. 质量打分、去重与 token 统计

In [ ]:
PYTHON_SIGNALS = (
    "def ", "class ", "import ", "from ", "return ", "yield ", "async ", "await ",
    "with ", "try:", "except ", "raise ", "pytest", "unittest", "typing", "__name__ ==",
)
CONTROL_CHARS_RE = re.compile("[\x00-\x08\x0b\x0c\x0e-\x1f]")
REPEATED_CHAR_RE = re.compile(r"(.)\1{24,}")


def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = CONTROL_CHARS_RE.sub("", text)
    return text.strip()


def stable_hash(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()


def quality_score(text: str, ds: dict[str, Any]) -> float:
    text = normalize_text(text)
    if len(text) < MIN_TEXT_CHARS:
        return 0.0
    lines = text.splitlines()
    non_empty = [line for line in lines if line.strip()]
    if not non_empty:
        return 0.0
    ascii_ratio = sum(ord(ch) < 128 for ch in text) / max(1, len(text))
    code_signal = sum(sig in text for sig in PYTHON_SIGNALS) / len(PYTHON_SIGNALS)
    code_line_ratio = sum(
        bool(re.match(r"\s*(def|class|import|from|return|if|for|while|try|except|with|@)", line))
        for line in non_empty
    ) / max(1, len(non_empty))
    avg_line_len = sum(len(line) for line in non_empty) / max(1, len(non_empty))
    long_line_ratio = sum(len(line) > 220 for line in non_empty) / max(1, len(non_empty))
    line_health = 1.0 - min(1.0, long_line_ratio * 3.0 + max(0.0, avg_line_len - 120) / 300)
    repeated_penalty = 0.25 if REPEATED_CHAR_RE.search(text) else 0.0
    size_score = 1.0 if 400 <= len(text) <= 80_000 else 0.65
    hint = float(ds.get("quality_hint", 0.5))
    ast_bonus = 0.0
    if len(text) <= 200_000:
        try:
            import ast
            ast.parse(text)
            ast_bonus = 0.10
        except Exception:
            ast_bonus = 0.0
    score = (
        0.22 * ascii_ratio
        + 0.22 * min(1.0, code_signal * 3.0)
        + 0.16 * min(1.0, code_line_ratio * 2.5)
        + 0.16 * line_health
        + 0.10 * size_score
        + 0.10 * hint
        + ast_bonus
        - repeated_penalty
    )
    return max(0.0, min(1.0, score))


def load_sample_summary(path: Path) -> list[dict[str, Any]]:
    out = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            rec = json.loads(line)
            rec.pop("text", None)
            out.append(rec)
    return out


def build_sample_store() -> list[dict[str, Any]]:
    if SAMPLES_JSONL.exists() and not FORCE_RESCAN:
        print(f"[scan] reuse {SAMPLES_JSONL}")
        return load_sample_summary(SAMPLES_JSONL)
    seen = set()
    summary = []
    with SAMPLES_JSONL.open("w", encoding="utf-8") as handle:
        for ds in DATASETS:
            ds_seen = ds_kept = 0
            pbar = tqdm(desc=f"scan: {ds['name']}", unit="sample")
            for raw_text in iter_dataset_texts(ds):
                ds_seen += 1
                text = normalize_text(raw_text)
                if len(text) < MIN_TEXT_CHARS or len(text) > MAX_TEXT_CHARS:
                    pbar.update(1)
                    continue
                h = stable_hash(text)
                if h in seen:
                    pbar.update(1)
                    continue
                score = quality_score(text, ds)
                if score < MIN_STORE_QUALITY:
                    pbar.update(1)
                    continue
                ids = tokenizer.encode(text, add_eos=ADD_EOS)
                rec = {
                    "dataset": ds["name"],
                    "repo_id": ds["repo_id"],
                    "stage_hint": ds.get("stage", "auto"),
                    "quality_hint": float(ds.get("quality_hint", 0.5)),
                    "quality_score": round(float(score), 6),
                    "n_chars": len(text),
                    "n_tokens": len(ids),
                    "hash": h,
                    "text": text,
                }
                handle.write(json.dumps(rec, ensure_ascii=False) + "\n")
                seen.add(h)
                compact = dict(rec)
                compact.pop("text")
                summary.append(compact)
                ds_kept += 1
                pbar.update(1)
            pbar.close()
            print(f"{ds['name']}: seen={ds_seen:,} kept={ds_kept:,}")
    return summary


summary = build_sample_store()
print(f"total kept samples={len(summary):,}")
print(f"total tokens={sum(r['n_tokens'] for r in summary):,}")

## 6. 查看 token 总量与数据集占比

In [ ]:
def dataset_stats(records: list[dict[str, Any]]) -> list[dict[str, Any]]:
    total_tokens = sum(r["n_tokens"] for r in records)
    grouped = {}
    for r in records:
        g = grouped.setdefault(
            r["dataset"],
            {"dataset": r["dataset"], "repo_id": r["repo_id"], "samples": 0, "chars": 0, "tokens": 0, "quality_sum": 0.0},
        )
        g["samples"] += 1
        g["chars"] += r["n_chars"]
        g["tokens"] += r["n_tokens"]
        g["quality_sum"] += r["quality_score"]
    rows = []
    for g in grouped.values():
        rows.append({
            "dataset": g["dataset"],
            "repo_id": g["repo_id"],
            "samples": g["samples"],
            "chars": g["chars"],
            "tokens": g["tokens"],
            "token_share": g["tokens"] / max(1, total_tokens),
            "avg_quality": g["quality_sum"] / max(1, g["samples"]),
        })
    return sorted(rows, key=lambda row: row["tokens"], reverse=True)


stats_rows = dataset_stats(summary)
try:
    import pandas as pd
    display(pd.DataFrame(stats_rows))
except Exception:
    for row in stats_rows:
        print(row)
print(f"TOTAL TOKENS = {sum(r['n_tokens'] for r in summary):,}")
print(f"TOTAL CHARS  = {sum(r['n_chars'] for r in summary):,}")

## 7. 选择 main / anneal 两阶段数据

默认导出 token 预算为 `main≈85%`、`anneal≈15%`。`stage='anneal'` 优先进退火集，`stage='main'` 优先进主训练集，`auto` 样本按质量分补充退火集。两个阶段不重叠。

In [ ]:
def choose_stage_splits(records: list[dict[str, Any]]) -> tuple[set[str], set[str], dict[str, Any]]:
    total_tokens = sum(r["n_tokens"] for r in records)
    target_anneal = int(total_tokens * ANNEAL_TOKEN_RATIO)
    target_main = int(total_tokens * MAIN_TOKEN_RATIO)

    anneal_forced = [r for r in records if r["stage_hint"] == "anneal"]
    auto_records = [r for r in records if r["stage_hint"] == "auto"]
    auto_sorted = sorted(auto_records, key=lambda r: r["quality_score"], reverse=True)
    top_auto_count = max(1, int(len(auto_sorted) * AUTO_ANNEAL_TOP_FRACTION)) if auto_sorted else 0
    high_quality_auto = [
        r for i, r in enumerate(auto_sorted)
        if r["quality_score"] >= QUALITY_SCORE_FOR_ANNEAL or i < top_auto_count
    ]
    ranked_anneal = sorted(
        anneal_forced + high_quality_auto,
        key=lambda r: (r["stage_hint"] == "anneal", r["quality_score"], r["n_tokens"]),
        reverse=True,
    )

    anneal_hashes = set()
    anneal_tokens = 0
    for r in ranked_anneal:
        if anneal_tokens >= target_anneal:
            break
        anneal_hashes.add(r["hash"])
        anneal_tokens += r["n_tokens"]

    remaining = [r for r in records if r["hash"] not in anneal_hashes]
    ranked_main = sorted(
        remaining,
        key=lambda r: (r["stage_hint"] == "main", r["quality_score"], r["n_tokens"]),
        reverse=True,
    )
    main_hashes = set()
    main_tokens = 0
    for r in ranked_main:
        if main_tokens >= target_main:
            break
        main_hashes.add(r["hash"])
        main_tokens += r["n_tokens"]

    info = {
        "total_tokens_scanned": total_tokens,
        "target_main_tokens": target_main,
        "target_anneal_tokens": target_anneal,
        "main_tokens_selected": main_tokens,
        "anneal_tokens_selected": anneal_tokens,
        "main_samples": len(main_hashes),
        "anneal_samples": len(anneal_hashes),
        "main_token_share_selected": main_tokens / max(1, main_tokens + anneal_tokens),
        "anneal_token_share_selected": anneal_tokens / max(1, main_tokens + anneal_tokens),
    }
    return main_hashes, anneal_hashes, info


main_hashes, anneal_hashes, split_info = choose_stage_splits(summary)
print(json.dumps(split_info, ensure_ascii=False, indent=2))

## 8. 导出 `main.bin` / `anneal.bin` 与元数据

In [ ]:
def remove_if_needed(path: Path, force: bool) -> None:
    if path.exists() and force:
        path.unlink()


def export_bins() -> dict[str, Any]:
    remove_if_needed(MAIN_BIN, FORCE_EXPORT_BINS)
    remove_if_needed(ANNEAL_BIN, FORCE_EXPORT_BINS)
    main_tokens = anneal_tokens = main_samples = anneal_samples = 0
    with (
        MAIN_BIN.open("ab") as main_f,
        ANNEAL_BIN.open("ab") as anneal_f,
        SAMPLES_JSONL.open("r", encoding="utf-8") as src,
        SPLIT_MANIFEST_JSONL.open("w", encoding="utf-8") as manifest,
    ):
        for line in tqdm(src, desc="export bins", unit="sample"):
            rec = json.loads(line)
            h = rec["hash"]
            if h in anneal_hashes:
                split = "anneal"
            elif h in main_hashes:
                split = "main"
            else:
                split = "unused"
            compact = {k: v for k, v in rec.items() if k != "text"}
            compact["split"] = split
            manifest.write(json.dumps(compact, ensure_ascii=False) + "\n")
            if split == "unused":
                continue
            ids = np.asarray(tokenizer.encode(rec["text"], add_eos=ADD_EOS), dtype=DTYPE)
            if split == "anneal":
                ids.tofile(anneal_f)
                anneal_tokens += int(ids.size)
                anneal_samples += 1
            else:
                ids.tofile(main_f)
                main_tokens += int(ids.size)
                main_samples += 1
    return {
        "main_bin": str(MAIN_BIN.relative_to(ROOT)),
        "anneal_bin": str(ANNEAL_BIN.relative_to(ROOT)),
        "dtype": str(DTYPE),
        "main_tokens": main_tokens,
        "anneal_tokens": anneal_tokens,
        "main_samples": main_samples,
        "anneal_samples": anneal_samples,
    }


export_info = export_bins()
meta = {
    "kind": "walkie_data",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "output_dir": str(OUTPUT_DIR.relative_to(ROOT)),
    "tokenizer": {
        "kind": TOKENIZER_KIND,
        "vocab_size": tokenizer.vocab_size,
        "path": str(TOKENIZER_PATH.relative_to(ROOT)),
        "add_eos_per_sample": ADD_EOS,
    },
    "dtype": str(DTYPE),
    "datasets": [
        {k: (str(v.relative_to(ROOT)) if isinstance(v, Path) else v) for k, v in ds.items()}
        for ds in DATASETS
    ],
    "dataset_stats": stats_rows,
    "split_policy": {
        "main_token_ratio_target": MAIN_TOKEN_RATIO,
        "anneal_token_ratio_target": ANNEAL_TOKEN_RATIO,
        "quality_score_for_anneal": QUALITY_SCORE_FOR_ANNEAL,
        "auto_anneal_top_fraction": AUTO_ANNEAL_TOP_FRACTION,
        "train_step_anneal_start_ratio": 0.8,
    },
    "split_info": split_info | export_info,
    "files": {
        "samples_jsonl": str(SAMPLES_JSONL.relative_to(ROOT)),
        "split_manifest_jsonl": str(SPLIT_MANIFEST_JSONL.relative_to(ROOT)),
        "main_bin": str(MAIN_BIN.relative_to(ROOT)),
        "anneal_bin": str(ANNEAL_BIN.relative_to(ROOT)),
    },
    "train_overrides": {
        "data.stages.main.bin": MAIN_BIN.as_posix(),
        "data.stages.main.dtype": str(DTYPE),
        "data.stages.anneal.bin": ANNEAL_BIN.as_posix(),
        "data.stages.anneal.dtype": str(DTYPE),
    },
}
META_PATH.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(export_info, ensure_ascii=False, indent=2))
print(f"wrote {META_PATH}")

## 9. 快速验证与训练命令

In [ ]:
main_mm = np.memmap(MAIN_BIN, dtype=DTYPE, mode="r")
anneal_mm = np.memmap(ANNEAL_BIN, dtype=DTYPE, mode="r")
print(f"main tokens   = {len(main_mm):,}")
print(f"anneal tokens = {len(anneal_mm):,}")

preview_n = min(160, len(main_mm))
if preview_n:
    print("\n[main preview decode]")
    print(tokenizer.decode(main_mm[:preview_n].tolist())[:800])

cmd = (
    "uv run --extra walkie python -m train.walkie_pretrain "
    "--config configs/train/pretrain_walkie.yaml "
    f"data.stages.main.bin={MAIN_BIN.as_posix()} "
    f"data.stages.main.dtype={DTYPE} "
    f"data.stages.anneal.bin={ANNEAL_BIN.as_posix()} "
    f"data.stages.anneal.dtype={DTYPE}"
)
print("\n[start training]")
print(cmd)